# Session 19 · Chat state & memory windows
## Turning a single call into a real conversation

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 11. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
Every notebook through Session 18 did the same shape of work: one message in, one reply out, done. Session 16's
classifier reads one ticket and returns one label; Session 18 attacks and defends that same one-shot call. That
shape is perfect for a classifier. It is useless for a chat assistant, because a chat assistant has to answer
*"what about for self-employed applicants?"* after *"what are your personal loan rates?"* — and the second
question is meaningless without the first. Ask a model that question with nothing else attached, and it has no
idea what "self-employed applicants" are being compared against, because an LLM API call is **stateless**: the
model remembers nothing of any earlier call, ever. What looks like "the assistant remembers" is not a model
capability at all — it is your application code resending the whole conversation so far, as a growing list of
role-tagged messages, every single time.

That one fact has three consequences, and they are the entire session. First, **cost grows with the length of
the conversation, not the relevance of the question** — a call at turn 10 pays for turns 1 through 9 even when
none of them matter to turn 10's question. Second, **there is a hard ceiling**, the context window, and past it
a call does not degrade gracefully, it fails outright. Third, **something has to give**, because no application
can resend an unbounded history forever — which is exactly why today ends with two different fixes, measured
honestly against each other rather than picking a "winner" by assumption.

Today's running example, BankAssist, is new — not the Session 16/18 support classifier continued. That is
deliberate: multi-turn memory is easiest to feel with a conversation that genuinely NEEDS an early turn to
answer a late one, and a classifier that labels one ticket and stops never creates that need.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Build a real **messages list** that grows, turn by turn | The silent assumption that "the model remembers" — it never does, your code does | A running transcript, not a memory |
| **Meter cost per turn**, not per conversation | A chatbot bill that climbs for reasons nobody can explain | Session 15's habit, now over time |
| Name the **hard ceiling**: the context window | A production outage that looks like a random failure | The size of the room, not a soft limit |
| Build and compare a **sliding window** and **summarization** | Picking a memory strategy by guess, then discovering the trade-off in production | Two real fixes, two real costs |
| Measure **recall**, precisely, before trusting either fix | Assuming a fix "basically works" without ever testing whether an old fact survives | Evidence, not vibes — Session 17's habit, applied to memory |

### Where this is used in industry
- **Banking (BFS):** a relationship-manager chatbot mid loan conversation — "what's the rate for self-employed?"
  five turns later needs to still know the applicant said "home loan" at turn 2. Get the memory strategy wrong
  and the bot re-asks questions the customer already answered, a top complaint driver in real deployed banking
  assistants.
- **Healthcare:** a patient-intake assistant that asks about symptoms across several messages needs the fourth
  answer to still make sense of the first three — and a context-window failure mid-intake is a real incident,
  not a curiosity.
- **Insurance and retail:** any multi-turn support or sales chat accumulates history fast; a bot that "forgets"
  what the customer already said reads as broken, however good any single reply is.
- **Any system billed per token:** a chat product's token bill is overwhelmingly a function of HOW conversation
  history is managed, not how good the prompts are — this is why "my chatbot got expensive" is almost always a
  memory-architecture question, not a pricing one.

### Why interviewers care
"How do you manage conversation memory in an LLM application?" is now a standard systems-design question. Be
ready for:
1. "Does the model remember previous messages on its own?" (No — every call is stateless. "Remembering" is
   your application resending history. This is the one fact the whole session hangs off.)
2. "Why does a chatbot's cost grow the longer a conversation runs?" (Because a naive implementation resends the
   entire history on every call — turn 20 pays for turns 1 through 19, whether or not they are relevant.)
3. "What's the trade-off between a sliding window and summarization?" (A window is cheap and flat but silently
   drops anything outside it, no warning. Summarization recovers most recall at the cost of one extra call per
   compression, and a new risk: the summary itself can be wrong. Today measures both, honestly, rather than
   picking one by assumption.)
4. "What's the real industry answer to long-term memory at scale?" (Retrieval-based memory — store turns in a
   vector database, pull back only what's relevant per query. Deliberately NOT built today; it is why Module 4,
   vector databases, exists immediately after this module.)
5. "How would you measure whether a memory strategy actually works?" (A scripted, repeatable probe with a known
   planted fact and a defined recall check — exactly today's `RECALL_PROBE`, the same evidence-over-vibes
   discipline Session 17 taught for prompt quality.)

Each is answered, with an example from this lab, in Part C.

<details><summary><b>▸ Read more: isn't this just "give the model a bigger context window"?</b></summary>

A bigger window buys more room, not a solution — cost still grows with every token resent, and every window,
however large, still has an edge. The two fixes today (and the retrieval-based memory Module 4 builds) exist
because "just use a bigger window" trades a cost problem for a bigger cost problem, later. The right question is
never "how big is the window", it is "what does this application actually need to remember, and for how long."
</details>

## A2. Where this session fits in the course

```
 S9   Tokens, context window, roles  ────────► system/user/assistant named, never built into a real list
 S11  Multi-provider calls ──────────────► roles become every call's message shape
 S14  System prompt as a contract ──────► one instruction, obeyed every call — but still one call
 S15  Token metering ─────────────────► cost measured per call, not guessed
 S16  Structured prompting ─────────► system prompt as a file; still one ticket in, one label out
  │
 S19  Chat state & memory windows ─► TODAY: the contract now persists across MANY calls
  │        a messages list that grows · metered every turn · a hard ceiling · two fixes, compared
  │
 S20  Function calling ────────────────► the model requests an action; YOUR code executes it
 S21-22  LangChain / LiteLLM ──────► the same round trip, inside a framework and across providers
 S23  FastAPI + SSE serving ───────► the loop goes live — served over the network, streamed
  │
 Module 4  Vector databases ─────► retrieval-based memory replaces today's hand-rolled window
 Module 6  Agents ────────────────► the ReAct loop is S20's tool round trip, repeated in a while-loop
 Module 12-13  Security/Cloud ─► harden S23's server: auth, rate limits, timeouts
```

**What you need from earlier sessions:** your `.env` with a working key (Session 11), the course repository
pulled and synced, and nothing from Session 16/18's `prompts/support_classifier/` — BankAssist is a new,
standalone example and those files stay untouched today.

## A3. The words you need today

---
**1. Conversation history (the messages list)**
- **Definition:** the growing list of role-tagged messages (`system`, `user`, `assistant`) that is resent, in
  full, on every call. This list IS the conversation, as far as the model is concerned — nothing else exists.
- **Analogy:** a running transcript that gets handed over again, in full, every time you ask one more question.
- **Comes back in:** S20 (tool calls join the same list); Module 6 agents.

---
**2. Turn**
- **Definition:** one user message plus the assistant's reply to it. `RECALL_PROBE` today scripts 17 turns.
- **Analogy:** one exchange in a conversation — you speak, they answer.
- **Comes back in:** every session in this module.

---
**3. Stateless call**
- **Definition:** the model has zero memory of any previous API call. "The assistant remembers" is not a
  property of the model; it is your code resending history.
- **Often confused with:** the model "getting smarter" when it uses a tool or sees more context — it does not;
  it is the same text predictor, just reading a longer input.
- **Comes back in:** S20, Module 6 (an agent is this same fact, with tool results added to the same list).

---
**4. Sliding window**
- **Definition:** a memory strategy that keeps only the most recent N messages and silently drops the rest.
  Flat cost forever; anything outside the window is gone, with no warning to anyone.
- **Analogy:** a rolling cutoff — like a security camera that only keeps the last hour of footage.
- **Comes back in:** Module 4 (replaced by retrieval-based memory, which decides what to keep by relevance, not
  recency alone).

---
**5. Summarization (compression)**
- **Definition:** a memory strategy that periodically compresses older turns into one short note, carried
  forward in place of the raw text. Recovers most recall, at the cost of one extra model call per compression.
- **Analogy:** a running minutes-of-the-meeting note, rewritten every so often instead of keeping every word
  anyone said.
- **Comes back in:** Module 4; S54 LLMOps (monitoring a summarization strategy in production).

---
**6. Recall**
- **Definition:** whether a LATER turn can correctly use a fact stated in an EARLIER turn — checked on the
  model's VISIBLE reply only, exactly like Session 17/18's "judge the exam, not the margin notes" discipline.
- **Often confused with:** "the model seems to understand the conversation" as a vague impression — recall
  today is a precise, scripted, pass/fail check: does one exact token appear in one exact reply, yes or no.
- **Comes back in:** Module 4 (RAG evaluation reuses exactly this idea — did the retrieved context actually
  make it into the answer).

---
**7. Context window**
- **Definition:** the hard ceiling on how many tokens a single call can hold, prompt and reply together. Past
  it, a call does not degrade, it fails outright.
- **Analogy:** the size of the room everyone has to fit in — once it's full, nobody else gets in, politely or
  otherwise.
- **Comes back in:** S20, S23 (a long-running served conversation hits this for real); Module 4, S38.

## A4. How today is built

```
            ┌──────────────────────────────────────────────────┐
            │                    chat_tasks.py                             │
            │  BANKASSIST_SYSTEM (standing contract)  RECALL_TOKEN          │
            │  RECALL_PROBE (17-turn scripted chat)  WINDOW_SIZE  SUMMARY_EVERY │
            └───────────────────────┬────────────────────────────────────────────┘
                                         │
            ┌─────────────────────────────────▼──────────────────────────────────────┐
            │                    chat_utils.py                              │
            │  append_turn()         → the entire "memory" mechanism           │
            │  slide_window() / summarize_older()  → the two fixes             │
            │  recall_succeeded()    → checks the VISIBLE reply only            │
            │  run_strategy() / compare_strategies()  → the full 17-turn runs    │
            │  chat_memory_markdown() → the report you commit                   │
            └───────────────────────┬────────────────────────────────────────────┘
                                         │
                              YOUR notebook sections 1–11
                                         │
                         experiments/s19_chat_memory.md  (committed)
```

Section order: feel the problem on a single stateless call → build a real messages list and watch it succeed
→ run the full 17-turn naive conversation → measure the cost growth, and project it further → fix it with a
sliding window → fix it with summarization → compare all three, honestly → save the report → checkpoint.

---
# Part B · Hands-on

## Section 1 · Setup
**What and why.** Load your key, build a `Lab` (Session 15's caching wrapper), and import today's two files.
Nothing here is new machinery — it is Session 15–18's setup, reused.

In [1]:
import sys
from pathlib import Path
# sys.path.insert(0, "prompting")
# sys.path.insert(0, "module03")
# sys.path.insert(0, "../prompting")

ROOT = Path.cwd() if (Path.cwd() / "prompting").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "prompting"))
sys.path.insert(0, str(ROOT / "module03"))

from dotenv import load_dotenv
load_dotenv(override=True)

import os
from reasoning_utils import DiskCache, Lab
import openai

client = openai.OpenAI(api_key=os.environ["API_KEY"], base_url=os.environ["BASE_URL"])
MODEL = os.environ["MODEL"]
cache = DiskCache(Path.home() / "Desktop" / "FSDS" / "gen_ai" / "applied-llm-ops" / "chat_memory_cache.json")
lab = Lab(client, MODEL, cache=cache)

from chat_tasks import BANKASSIST_SYSTEM, RECALL_PROBE, RECALL_TOKEN, SUMMARY_EVERY, WINDOW_SIZE
from chat_utils import (append_turn, chat_memory_markdown, check_chat_memory_work, compare_strategies, parse_chat_report,
                        recall_succeeded, run_strategy, save_chat_report, slide_window, strategy_comparison_table,
                        summarize_older, turn_cost_table)

print(f"model: {MODEL}")
print(f"conversation length: {len(RECALL_PROBE)} turns")
print(f"planted fact (recall token): {RECALL_TOKEN!r}, planted at turn 2, asked for again at turn {len(RECALL_PROBE)}")

model: openai/gpt-oss-20b
conversation length: 17 turns
planted fact (recall token): '77042', planted at turn 2, asked for again at turn 17


**What you should see.** The model name, `17`, and the recall token `'77042'`.

**Check yourself.** If you get a `KeyError` on `API_KEY`, your `.env` is missing or not in the current folder —
re-check Session 11.

## Section 2 · A stateless call has no memory — see it fail
**What and why.** Before building anything, prove the one fact the whole session depends on: send the RECALL
QUESTION (turn 17's exact text) as a single, stateless call — nothing else attached, no history, not even
turn 2 where the reference number was given. If "the model remembers" were a real model capability, it might
guess or recover somehow. It cannot, because there is nothing to recover — the fact was never part of this
call.

**The idea.** This is deliberately the harshest possible test: one system message, one user message (the
recall question), nothing else. Watch it fail, cleanly, every time — not because the model is bad at
remembering, but because there is nothing here TO remember.

In [2]:
stateless_messages = [
    {"role": "system", "content": BANKASSIST_SYSTEM},
    {"role": "user", "content": RECALL_PROBE[-1]},   # the recall question, alone, with no history
]
r = lab.chat(stateless_messages)
print(r["text"])
print()
print("recall token present:", recall_succeeded(r["text"]))

I’m sorry, but I don’t have that reference number on hand. I can connect you to a relationship manager who can look it up for you.

recall token present: False


**What you should see.** A polite reply that does NOT contain `77042` — the model will likely say it does
not have a reference number on file, or ask the customer to repeat it. `recall_succeeded` should print `False`.
This is not a weak model; this is the correct behaviour of a stateless call that was never told the number.

**Check yourself.** Read `BANKASSIST_SYSTEM` (print it if you like). Confirm for yourself that the reference
number `77042` does not appear anywhere in the system prompt — it can ONLY come from the conversation itself.

## Section 3 · Build a real messages list — see it succeed
**What and why.** `append_turn` is the entire fix: a NEW message added to a NEW list, which is then resent on
the NEXT call. Walk turns 1 and 2 by hand, then immediately ask the recall question (skipping the 14 filler
turns for now) — this isolates ONE variable: does having turn 2 physically present in the call change the
answer.

**What the next cell does.** 1) Start the messages list with just the system prompt. 2) Append turn 1, call the
model, append its reply. 3) Append turn 2 — the one with the reference number — call the model, append its
reply. 4) Append the recall question and call the model one more time, with turns 1 and 2 still sitting in the
list.

In [3]:
messages = [{"role": "system", "content": BANKASSIST_SYSTEM}]

messages = append_turn(messages, "user", RECALL_PROBE[0])
r1 = lab.chat(messages)
messages = append_turn(messages, "assistant", r1["text"].strip())

messages = append_turn(messages, "user", RECALL_PROBE[1])   # plants the reference number
r2 = lab.chat(messages)
messages = append_turn(messages, "assistant", r2["text"].strip())

messages = append_turn(messages, "user", RECALL_PROBE[-1])  # ask for it back, right away
r3 = lab.chat(messages)
print(r3["text"])
print()
print("recall token present:", recall_succeeded(r3["text"]))
print("messages in the call:", len(messages) + 1)   # +1 for the question just appended

Your reference number is LN‑77042‑BLR.

recall token present: True
messages in the call: 7


**What you should see.** The reply should now contain `77042` — `recall_succeeded` prints `True`. Nothing
about the MODEL changed between Section 2 and this cell; the only thing that changed is that turn 2 is
physically present in this call's `messages` list. That is the whole mechanism, in full, right there.

**Check yourself.** Print `len(messages)` before the last `append_turn` call. If "remembering" were a model
feature, this number would not matter. It is the only thing that matters.

## Section 4 · Run the full 17-turn conversation, naively
**What and why.** Section 3 skipped the 14 filler turns. Real conversations do not skip anything — `run_strategy`
plays the ENTIRE scripted `RECALL_PROBE`, turn by turn, with the `"naive"` strategy: every turn's call includes
every earlier turn, no trimming at all. If naive history is going to recall the reference number at turn 17, it
has to do it with 14 unrelated FAQ questions sitting in between.

**What the next cell does.** `run_strategy` builds the messages list fresh, plays all 17 turns, and returns the
per-turn cost, the final reply, and whether it recalled the planted fact.

In [4]:
naive_result = run_strategy(lab, "naive")

print(naive_result["final_reply"])
print()
print("recall succeeded:", naive_result["recall_ok"])
print("turns played:", naive_result["n_turns"])

DailyLimit: The provider says to wait about 8 minutes. That is a daily limit, not a per-minute one: stop here, or carry on tomorrow. Answers already received are saved, so nothing is lost.

**What you should see.** `recall succeeded: True` — naive history keeps EVERYTHING, so the reference number
from turn 2 is still sitting in the call at turn 17, however many unrelated questions came in between.

**Check yourself.** If this says `False`, something is broken in the setup (not the memory strategy) — check
that `RECALL_TOKEN` and `RECALL_PROBE` in `chat_tasks.py` were not edited, and that `BANKASSIST_SYSTEM` still
instructs the model to remember what the customer tells it about themselves.

## Section 5 · Measure the cost growth
**What and why.** `recall_succeeded` was the first half of today's story. The second half is what naive history
COSTS to get that right. `turn_costs` is the input+output token count of every single one of the 17 calls —
read it as a sequence, not a single number, because the shape of the growth is the entire point.

In [ ]:
print(turn_cost_table(naive_result))
print()
first, last = naive_result["turn_costs"][0], naive_result["turn_costs"][-1]
print(f"turn 1 cost: {first:,} tokens")
print(f"turn {naive_result['n_turns']} cost: {last:,} tokens")
print(f"growth: {last / first:.1f}x, for a conversation that only grew {naive_result['n_turns']}x in NUMBER of turns")

**What you should see.** A climbing column of numbers, not a flat one — the real, measured shape behind the
deck's illustration. The growth ratio at the end will usually be smaller than the turn-count ratio (because
earlier turns are often shorter than later ones, not because the growth isn't real), but it never goes DOWN.

**Check yourself.** Is any later turn's cost LOWER than turn 1's? It should not be — naive history never drops
anything, so the list can only grow.

## Section 6 · Push it further: where does this actually break?
**What and why.** 17 turns is not where naive history breaks — it is where you can afford to watch it start
climbing without running out of budget. Project the SAME per-turn growth rate forward using the numbers you
just measured, not invented ones, to see where a real, long-running assistant (hours of conversation, not 17
turns) would actually hit trouble.

**What the next cell does.** Fits the measured cost growth to a straight line and reports the projected cost at
turn 100 — a number from YOUR run, not a guess.

In [ ]:
costs = naive_result["turn_costs"]
n = len(costs)
avg_growth_per_turn = (costs[-1] - costs[0]) / (n - 1)
projected_turn_100 = costs[-1] + avg_growth_per_turn * (100 - n)

print(f"measured growth: about {avg_growth_per_turn:,.0f} tokens per additional turn")
print(f"projected cost at turn 100, same rate: about {projected_turn_100:,.0f} tokens for ONE call")
print()
print("Two separate walls, in order: (1) that single call gets expensive long before any ceiling is reached —")
print("real, compounding spend, every call, on a long-running assistant. (2) every model ALSO has a hard token")
print("ceiling (check your provider's model page for this model's exact number) — past it, a call does not")
print("degrade, it fails outright, with no partial answer.")

**What you should see.** A projected number clearly larger than turn 17's real cost — the trend, extrapolated
honestly from your own measurement, not asserted. This is the number a real production assistant left running
for hours would actually reach.

**Check yourself.** This projection assumes the SAME growth rate holds (roughly true for similar-length
messages). A conversation with longer messages later on would climb even faster — the projection here is
therefore if anything optimistic, not pessimistic.

## Section 7 · Fix 1: a sliding window
**What and why.** `slide_window` keeps the system message, plus only the most recent `WINDOW_SIZE` messages
after it — everything older is silently dropped, every single call. Run the full 17-turn script again, this
time with the `"window"` strategy, and watch both halves of the trade-off: the cost (should go flat) and the
recall (should fail, by design — turn 2 is long outside a 6-message window by turn 17).

In [ ]:
window_result = run_strategy(lab, "window", window_size=WINDOW_SIZE)

print(window_result["final_reply"])
print()
print("recall succeeded:", window_result["recall_ok"])
print()
print(turn_cost_table(window_result))

**What you should see.** `recall succeeded: False` in almost every run — WINDOW_SIZE=6 messages means only
the last few exchanges survive, and turn 2 is 15 turns back by turn 17. The cost column should flatten out once
the window fills, unlike Section 5's climbing column.

**Check yourself.** Compare `window_result['turn_costs'][-1]` to `naive_result['turn_costs'][-1]`. The window's
final-turn cost should be noticeably lower — that gap IS what the window bought you, and the recall failure
above is exactly what it cost.

## Section 8 · Fix 2: summarization
**What and why.** `summarize_older` takes the turns a window would discard and spends ONE extra model call
compressing them into a short note — kept forward instead of the raw text. Run the script again with the
`"summarize"` strategy. The hope is that the reference number survives INSIDE the summary even after the
original turn 2 text is gone from the message list.

In [ ]:
summarize_result = run_strategy(lab, "summarize", window_size=WINDOW_SIZE, summary_every=SUMMARY_EVERY)

print(summarize_result["final_reply"])
print()
print("recall succeeded:", summarize_result["recall_ok"])
print("extra model calls spent summarizing:", summarize_result["extra_calls"])
print()
print(turn_cost_table(summarize_result))

**What you should see.** `recall succeeded` is usually `True` — the summarizing call was explicitly asked to
keep reference numbers and figures exact, so the planted fact typically survives into the note. `extra_calls`
should be greater than zero (one call every `SUMMARY_EVERY` turns). The final-turn cost should sit between the
window's and naive's.

**Check yourself.** If `recall succeeded` is `False` here, that is not a bug to fix — it is a genuine, reportable
finding: the summarizing call dropped or garbled the one detail that mattered. Print the summary note itself
(look inside `summarize_result['messages']` for the `role: "system"` entry) and read what it actually kept.

## Section 9 · Compare all three, side by side
**What and why.** Same script, same model, only the memory strategy changed between runs — any difference below
is attributable to the strategy, not to chance. `compare_strategies` re-runs all three (the earlier two calls
are free: `DiskCache` already has them).

In [ ]:
results = compare_strategies(lab, window_size=WINDOW_SIZE, summary_every=SUMMARY_EVERY)
print(strategy_comparison_table(results))

**What you should see.** Naive: highest cost, recall `yes`. Window: lowest cost, recall usually `no`.
Summarize: cost in between, recall usually `yes`, `extra calls` greater than zero. No strategy should win on
BOTH cost and recall at once — that tension is today's actual lesson, not a puzzle to resolve.

**Check yourself.** If one strategy somehow wins both columns on your run, re-read Section 8's note about
genuine findings — report what you saw rather than assuming it must be wrong.

## Section 10 · Save the report (your checkpoint)
**What and why.** `chat_memory_markdown` builds the comparison you just read into one file; `save_chat_report`
writes it to `experiments/s19_chat_memory.md` in your **own** work repository (`genai-course-work`), pushed
there, never to the shared course repository.

In [ ]:
WORK_REPO = r"C:\Users\HP\Desktop\FSDS\gen_ai\applied-llm-ops"   # <-- change this to YOUR folder

report_text = chat_memory_markdown(results, MODEL)
print(report_text)

In [ ]:
SAVE_REPORT = False   # flip to True once you are happy with the numbers above

if SAVE_REPORT:
    path = save_chat_report(WORK_REPO, report_text)
    print(f"saved: {path}")

## Section 11 · Checkpoint
**What and why.** `check_s19_work` reads your saved report back and checks it offline: all three strategies
scored, naive recall succeeding (a sanity check the setup itself is correct), and both fixes costing less than
naive at the final turn.

In [ ]:
from prompt_utils import results_table

checkpoint_results = check_chat_memory_work(WORK_REPO)
print(results_table(checkpoint_results))

**What you should see.** Every line `PASS`. A `FAIL` names exactly what to fix and where (go back to Section
9 or 10). Commit and push `experiments/s19_chat_memory.md` in your `genai-course-work` repository once every
line is `PASS` — this is the deliverable the live checkpoint test reads.

---
# Part C · Wrap-up

## C1. Summary
- An LLM call is **stateless** — full stop. "The assistant remembers" is not a model capability; it is your
  application code resending the whole conversation, as a growing `messages` list, on every single call.
- That one fact has three consequences: **cost grows with conversation length, not relevance**; there is a
  **hard ceiling**, the context window, past which a call fails outright rather than degrading; and **something
  has to give**, because no application can resend an unbounded history forever.
- A **sliding window** buys a flat cost ceiling by silently dropping anything outside it — no warning, ever.
- **Summarization** recovers most recall at the price of one extra model call per compression, and a new risk:
  the summary itself can be wrong.
- Neither fix is free, and no strategy wins on both cost and recall at once — that tension is a real
  engineering trade-off to make deliberately, not a problem this session solves once and for all.
- A third fix, **retrieval-based memory** via a vector database, is deliberately not built today — that gap is
  exactly why Module 4 exists immediately after this module.

## C2. Honest limits — read this before you trust either fix in production
Today solved cost growth, not memory, perfectly. Three things named here, not fully solved:
- **Silent forgetting.** A windowed assistant never SAYS it dropped something — it just answers as if the
  earlier turn never happened. That can look confidently wrong rather than visibly uncertain, which is worse
  for a customer than an honest "I don't have that anymore."
- **Summarization can compound its own errors.** If the compressing call misstates or drops a fact, every later
  turn inherits that mistake, and nothing in today's pipeline catches it — the error propagates rather than
  self-corrects.
- **Untuned knobs.** `WINDOW_SIZE=6` and `SUMMARY_EVERY=6` were picked to make today's point visible in one
  17-turn script, not validated against a real support conversation's actual length and recall needs. A real
  application tunes these against its own traffic, not a classroom default.

These return properly in Module 4 (retrieval-based memory, evaluated the same evidence-over-vibes way) and
Module 12 (monitoring a memory strategy once it's in production).

## C3. Interview questions, answered
1. **Does the model remember previous messages on its own?** No. Every API call is stateless. "Remembering" is
   entirely your application resending the conversation history — Section 2's cell proves this directly: the
   same model, the same question, fails with no history and succeeds with it.
2. **Why does a chatbot's cost grow the longer a conversation runs?** Because a naive implementation resends
   the ENTIRE history on every call. Turn 20's call pays for turns 1 through 19 whether or not they are
   relevant to turn 20's question — Section 5 measures this directly rather than asserting it.
3. **What's the trade-off between a sliding window and summarization?** A window is cheap and flat but silently
   drops anything outside it, with zero warning to anyone. Summarization recovers most of that recall, at the
   cost of one extra model call per compression and a new failure mode: the summary itself can be wrong.
   Section 9 measures both, side by side, rather than assuming one is simply "better."
4. **What's the real industry answer to long-term memory at scale?** Retrieval-based memory: store turns in a
   vector database and pull back only what's relevant to the current query, rather than keeping everything or
   blindly dropping anything. That is deliberately not built today — it is Module 4's entire subject.
5. **How would you measure whether a memory strategy actually works?** A scripted, repeatable probe with one
   precisely defined success check, decided BEFORE looking at results — exactly `RECALL_PROBE` and
   `recall_succeeded`, Session 17's evidence-over-vibes discipline applied to memory instead of quality.

## C4. Common mistakes
- **Assuming the model "just remembers" short conversations.** It never does — even a two-turn conversation
  only works because your code resent turn 1 when it called the model for turn 2. Section 2 vs 3 is the proof.
- **Treating cost as flat because a single call "feels the same."** Cost is a property of the WHOLE history
  resent, not of the question just asked — Section 5's climbing column is the evidence, not an assertion.
- **Picking a window size or compression schedule without testing recall.** A window that is "probably fine"
  can silently fail on exactly the fact a customer needed — Section 7's `recall succeeded: False` is there on
  purpose, to make this concrete before it happens in production.
- **Trusting a summary without checking what it actually kept.** Section 8's note to inspect the summary's own
  text directly is not optional — a summarizing call can drop the one number that mattered while still
  "sounding" complete.

## C5. Troubleshooting
| Symptom | Likely cause | Fix |
|---|---|---|
| Section 2's stateless call somehow recalls `77042` | A leftover `DiskCache` entry from a different run, or `BANKASSIST_SYSTEM` was edited to include the number | Delete `~/.genai_course/s19_cache.json` and re-run; check `chat_tasks.py` is unchanged |
| Section 4's naive recall fails | `RECALL_TOKEN` or `RECALL_PROBE` was edited in `chat_tasks.py` | Re-pull the course repo, or check your edits did not remove the reference number from turn 2 |
| Section 7's window recall unexpectedly SUCCEEDS | `WINDOW_SIZE` was set too large for a 17-turn script | Lower `WINDOW_SIZE`, or treat it as a genuine finding and say so in your report |
| Section 8's summarize `extra_calls` is 0 | `SUMMARY_EVERY` is larger than 17, so compression never triggers | Lower `SUMMARY_EVERY` below the script length |
| `check_s19_work` fails on "naive strategy ... recalls" | Something broke the baseline itself, not a real strategy result | Re-run Section 4 alone first; fix before touching the fixes |

## C6. Assignment
Add a FOURTH strategy, `"window_plus_summary"`, that keeps a small sliding window AND periodically summarizes
everything older than the window (rather than summarization's current all-or-nothing split). Re-run
`compare_strategies` with your new strategy included and report, honestly, whether it beats either of today's
two fixes on both cost and recall — or whether, like today's two, it only wins on one axis. A memory strategy's
real trade-offs are a living design decision, not a one-time fix — this is the exercise that makes that
concrete.